这是对第二次Kaggle，树叶分类竞赛的一些尝试

In [ ]:
import os  # 导入操作系统接口模块，用于文件路径操作、目录管理等, 所以关于文件的操作都可以使用这个模块
import numpy as np  
import pandas as pd
from PIL import Image # 从PIL（Python Imaging Library）导入Image模块，用于图像读取和处理
from tqdm import tqdm # 导入进度条工具，用于显示循环或处理的进度

import torch  
import torch.nn as nn # 网络层
import torch.optim as optim # 优化器
from torch.utils.data import Dataset, DataLoader  # 导入数据集类和数据加载器，用于批量加载训练数据
from torchvision import transforms, models  # 导入图像变换工具和预训练模型
from sklearn.model_selection import train_test_split  # 从scikit-learn导入数据集分割函数，用于划分训练集和测试集

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [ ]:
data_dir = '../chapter_convolutional-modern' # 设置数据所在的目录路径, 指向存放数据文件的文件夹

# os.path.join() 函数用于将多个路径组合成一个完整的路径, 这样可以确保在不同操作系统上路径的正确性
# pd.read_csv() 使用Pandas读取CSV文件，返回DataFrame（数据表）
train_df = pd.read_csv(os.path.join(data_dir, 'train.csv')) 
test_df  = pd.read_csv(os.path.join(data_dir, 'test.csv'))

print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')

# 标签编码
labels    = sorted(train_df['label'].unique().tolist())
# train_df['label'] 是获取训练集中的'label'列,.unique()是获取所有不重复的标签值,
# .tolist()是将这些标签值转换为列表, sorted()是对标签列表进行排序
label2idx = {label: idx for idx, label in enumerate(labels)}
# enumerate(labels) 会返回一个索引和值的迭代器, 这里将标签映射为索引, 形成一个字典label2idx, 其中键是标签, 值是对应的索引
# 这一步是将文本标签转换为数字索引, 便于模型训练和预测
idx2label = {idx: label for label, idx in label2idx.items()}  # 预测后将数字结果转回文本标签
num_classes = len(labels) 
print(f'Number of classes: {num_classes}')

train_df['label_idx'] = train_df['label'].map(label2idx)  
# train_df['label'].map(label2idx) 是将训练集中的文本标签映射为对应的数字索引, 并将结果存储在新的列'label_idx'中

Train size: 18353, Test size: 8800
Number of classes: 176


In [ ]:
class LeavesDataset(Dataset):
    def __init__(self, df, data_dir, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)  # 重置DataFrame的索引为0, 1, 2...,确保索引从0开始连续，方便后续访问
        self.transform = transform  # 保存图像变换操作
        self.is_test   = is_test   # 保存是否为测试集的标志

        # 预加载所有图片到内存，同时统一 resize 到 256 减少运行时开销
        print(f'Preloading {len(self.df)} images...')
        preload_tf = transforms.Resize(256)
        self.images = []
        for idx in tqdm(range(len(self.df))):
            img_path = os.path.join(data_dir, self.df.loc[idx, 'image'])
            img = Image.open(img_path).convert('RGB')
            img = preload_tf(img)
            self.images.append(img.copy())  # .copy() - 创建图片的副本,避免引用问题，确保每张图片独立存储
        print('Done ✅')

    def __len__(self):
        return len(self.df) 

    def __getitem__(self, idx):
        image = self.images[idx]
        if self.transform:
            image = self.transform(image)
        if self.is_test:
            return image
        label = self.df.loc[idx, 'label_idx']  # 从DataFrame中获取对应的数字标签
        return image, label

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([   # Compose() 将多个变换操作组合成一个流水线,按顺序依次执行每个变换
    transforms.RandomCrop(IMG_SIZE),       # 数据增强 - 每次训练看到的是图片的不同部分,随机裁剪
    transforms.RandomHorizontalFlip(),  # 随机水平翻转(50%概率)
    transforms.RandomVerticalFlip(),  # # 随机垂直翻转(50%概率)
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),  # 模拟不同光照条件
    transforms.ToTensor(), # 转换为神经网络可以处理的格式 会自动归一化到[0, 1]
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),    
    # Z-score归一化 第一个列表为三个通道的均值，第二个列表为三个通道的标准差, 这是ImageNet数据集的统计值, 适用于大多数预训练模型
])

val_transform = transforms.Compose([
    transforms.CenterCrop(IMG_SIZE),       # 不需要 Resize 了
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_data, val_data = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['label_idx']   # stratify=train_df['label_idx']分层采样，保持各类别比例
)

train_dataset = LeavesDataset(train_data, data_dir, transform=train_transform)  # 训练集使用 train_transform（有数据增强）
val_dataset   = LeavesDataset(val_data,   data_dir, transform=val_transform) # 验证集使用 val_transform（无数据增强）
test_dataset  = LeavesDataset(test_df,    data_dir, transform=val_transform, is_test=True) # 测试集使用 val_transform + is_test=True（返回图片，不返回标签）

# 创建数据加载器  pin_memory=True  提升数据传输到GPU的速度
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

Preloading 14682 images...


  0%|          | 0/14682 [00:00<?, ?it/s]

100%|██████████| 14682/14682 [02:29<00:00, 98.20it/s] 


Done ✅
Preloading 3671 images...


100%|██████████| 3671/3671 [00:36<00:00, 99.55it/s] 


Done ✅
Preloading 8800 images...


100%|██████████| 8800/8800 [00:22<00:00, 395.39it/s]

Done ✅
Train: 14682, Val: 3671, Test: 8800


In [ ]:
model = models.resnet18(weights='IMAGENET1K_V1')  # 从torchvision加载ResNet18架构，加载在ImageNet数据集上预训练的权重
model.fc = nn.Linear(model.fc.in_features, num_classes) 
# 因为原本是输出1000类的，明显与我们的任务不符，所以我们需要修改最后的全连接层，使其输出类别数为num_classes
model = model.to(device)
print('ResNet18 loaded ✅')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\13422/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [01:56<00:00, 403kB/s]


ResNet18 loaded ✅


In [ ]:
EPOCHS = 20

criterion = nn.CrossEntropyLoss()  # 专门用于多分类问题的交叉熵损失函数
optimizer = optim.Adam(model.parameters(), lr=1e-3) # 定义Adam优化器，学习率为1e-3，优化模型的所有参数(model.parameters())
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6) 
# 余弦退火学习率调度器,动态调整学习率，不是固定的1e-3
# T_max=EPOCHS 意味着学习率在20个epoch内完成一个周期,eta_min=1e-6是最小学习率，学习率不会降到这个值以下

In [ ]:
best_val_acc = 0.0
save_path    = 'best_resnet18.pth'

for epoch in range(1, EPOCHS + 1):
    # 训练
    model.train()
    train_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} Train', leave=False):
        # 从DataLoader批量获取数据 leave=False 完成后不保留进度条（节省屏幕空间）
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)  # 该batch的平均损失 ✖ 批量大小 
        correct    += (outputs.argmax(1) == labels).sum().item()  # outputs.argmax(1)是找到最大的索引，1是dim，总的来说就是找到对的数量
        total      += images.size(0)  # 该批次大小
    train_loss /= total
    train_acc   = correct / total

    # 验证
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            correct  += (outputs.argmax(1) == labels).sum().item()
            total    += images.size(0)
    val_loss /= total
    val_acc   = correct / total

    scheduler.step()  # 每个epoch结束后调整学习率

    is_best = val_acc > best_val_acc  # 生成一个bool值
    if is_best:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)  #  返回包含所有参数的字典

    print(f'Epoch [{epoch:02d}/{EPOCHS}]  '
          f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  |  '
          f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}  '
          f'{"★ Best" if is_best else ""}')
   # epoch:02d，表示两位数字，不足补0（01, 02, ...）
print(f'\nBest Val Acc: {best_val_acc:.4f}')

Epoch [01/20]  Train Loss: 2.9257  Train Acc: 0.2672  |  Val Loss: 2.2090  Val Acc: 0.3786  ★ Best


Epoch [02/20]  Train Loss: 1.7170  Train Acc: 0.4897  |  Val Loss: 1.6411  Val Acc: 0.5170  ★ Best


Epoch [03/20]  Train Loss: 1.2929  Train Acc: 0.5999  |  Val Loss: 1.2213  Val Acc: 0.6301  ★ Best


Epoch [04/20]  Train Loss: 1.0339  Train Acc: 0.6731  |  Val Loss: 0.8946  Val Acc: 0.7167  ★ Best


Epoch [05/20]  Train Loss: 0.8314  Train Acc: 0.7338  |  Val Loss: 0.7664  Val Acc: 0.7507  ★ Best


...

Epoch [16/20]  Train Loss: 0.1284  Train Acc: 0.9570  |  Val Loss: 0.1783  Val Acc: 0.9346  ★ Best


Epoch [17/20]  Train Loss: 0.1091  Train Acc: 0.9649  |  Val Loss: 0.1651  Val Acc: 0.9398  ★ Best


Epoch [18/20]  Train Loss: 0.0972  Train Acc: 0.9687  |  Val Loss: 0.1590  Val Acc: 0.9428  ★ Best


Epoch [19/20]  Train Loss: 0.0923  Train Acc: 0.9710  |  Val Loss: 0.1544  Val Acc: 0.9442  ★ Best


Epoch [20/20]  Train Loss: 0.0849  Train Acc: 0.9735  |  Val Loss: 0.1529  Val Acc: 0.9428  

Best Val Acc: 0.9442


In [ ]:
model.load_state_dict(torch.load(save_path, map_location=device))   # map_location=device指定参数加载到哪个设备
model.eval()

preds = []
with torch.no_grad():
    for images in tqdm(test_loader, desc='Predicting'):  # desc= 用于设置进度条前面显示的文字描述。
        images  = images.to(device)
        outputs = model(images)
        preds.extend(outputs.argmax(1).cpu().numpy())

pred_labels = [idx2label[i] for i in preds]  # 将索引转换为标签名

submission = pd.DataFrame({         
    'image': test_df['image'],
    'label': pred_labels
})
submission.to_csv('submission.csv', index=False)  # 保存文件 不保存行索引，不然前面会多一列，为索引
print('✅ submission.csv saved!')
print(submission.head())

C:\Users\13422\AppData\Local\Temp\ipykernel_9848\2800637138.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path, map_location=devi

✅ submission.csv saved!
              image                label
0  images/18353.jpg      asimina_triloba
1  images/18354.jpg         betula_nigra
2  images/18355.jpg  platanus_acerifolia
3  images/18356.jpg       pinus_bungeana
4  images/18357.jpg  platanus_acerifolia
